# Finnish ASR — Whisper-small fine-tuning (Team Tokens)
COE025 · built for Colab's free T4.

**Pipeline:** load data -> baseline (Whisper-small zero-shot) -> fine-tune -> evaluation (test_cv / test_parl / test_fleurs, reported separately, WER + CER).

**Before running:** in Colab, set `Runtime > Change runtime type > T4 GPU`.

> **normalize() note:** the `normalize()` function below must be **identical** to the one in the data-preparation notebook (`scripts/prepare_data.ipynb`), so that the same text normalization is applied to every system. If it differs, WER/CER will be inflated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE = '/content/drive/MyDrive/finnish_asr'
os.makedirs(SAVE, exist_ok=True)

In [ ]:
# 1) Install dependencies
!pip -q install -U transformers datasets jiwer librosa accelerate evaluate soundfile

In [ ]:
# 2) Imports + normalize()
import re, torch, jiwer, unicodedata
import numpy as np
from datasets import load_dataset, Audio, concatenate_datasets

# normalize() — identical to the 7 steps in scripts/prepare_data.ipynb
_TAG_RE = re.compile(r"\[[^\]]*\]|<[^>]*>|\([^)]*\)$")

def normalize(text):
    if text is None:
        text = ""
    text = unicodedata.normalize("NFKC", text)                   # 1) NFKC
    text = _TAG_RE.sub(" ", text)                                # 2) strip [noise] <unk> (utt_id)
    text = text.lower()                                          # 3) lowercase
    text = re.sub(r"(?<=\d)[ .](?=\d{3}(?:\D|$))", "", text)     # 4) "1 500"/"1.500" -> "1500"
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)       # 5) punctuation -> space (ä ö å kept)
    text = text.replace("_", " ")                                # 6) underscore -> space
    text = re.sub(r"\s+", " ", text).strip()                     # 7) collapse extra whitespace
    return text

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# 3) Load dataset (downloads from the Hugging Face Hub automatically)
REPO = "zehraoz36/team-tokens-finnish-asr-data"
splits = ["train", "dev_cv", "dev_parl", "test_cv", "test_parl", "test_fleurs"]
ds = {s: load_dataset(REPO, split=s) for s in splits}

for s in splits:
    ds[s] = ds[s].cast_column("audio", Audio(sampling_rate=16000))

# auto-detect the text column name (text / sentence / transcription)
cols = ds["train"].column_names
TEXT_COL = next(c for c in ["text","sentence","transcription","transcript"] if c in cols)
print("Columns:", cols, "| text column:", TEXT_COL)
for s in splits: print(f"{s:12s}: {len(ds[s])} clips")

In [ ]:
# Speaker-leak check
def spk_col(d):
    for c in ["speaker_id","client_id","speaker","spk_id","session_id"]:
        if c in d.column_names: return c
    return None

col = spk_col(ds["train"])
print("Speaker column:", col)

if col:
    def spk(split): return set(ds[split][col])
    tr = spk("train")
    for s in ["dev_cv","dev_parl","test_cv","test_parl","test_fleurs"]:
        shared = tr & spk(s)
        print(f"train ∩ {s}: {len(shared)} shared speakers")
else:
    print("No speaker ID column in this dataset; speaker-level split cannot be verified here.")

In [ ]:
# 4) Model + processor
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="finnish", task="transcribe")
tokenizer = processor.tokenizer

def load_model():
    m = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)
    m.generation_config.language = "finnish"
    m.generation_config.task = "transcribe"
    m.generation_config.forced_decoder_ids = None
    return m

In [ ]:
# 5) Evaluation function (progress bar on, warnings silenced)
import warnings, logging
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

@torch.no_grad()
def evaluate_split(model, dset, batch_size=16):
    model.eval()
    if DEVICE == "cuda": model.half()
    refs, hyps = [], []
    for i in tqdm(range(0, len(dset), batch_size), desc="eval"):
        batch = dset[i:i+batch_size]
        arrays = [a["array"] for a in batch["audio"]]
        feats = processor.feature_extractor(
            arrays, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
        if DEVICE == "cuda": feats = feats.half()
        gen = model.generate(feats, max_new_tokens=225)
        pred = processor.batch_decode(gen, skip_special_tokens=True)
        hyps += [normalize(p) for p in pred]
        refs += [normalize(t) for t in batch[TEXT_COL]]
    return 100*jiwer.wer(refs, hyps), 100*jiwer.cer(refs, hyps)

def report(model, tag):
    rows = []
    for s in ["test_cv", "test_parl", "test_fleurs"]:
        print(f"-> evaluating {s} ({len(ds[s])} clips)...")
        wer, cer = evaluate_split(model, ds[s])
        rows.append((s, wer, cer))
        print(f"[{tag}] {s:12s}  WER={wer:5.2f}  CER={cer:5.2f}")
    return rows

In [ ]:
# 6) BASELINE — Whisper-small zero-shot (no training)
base_model = load_model()
baseline_rows = report(base_model, "baseline")
del base_model; torch.cuda.empty_cache()

In [ ]:
# 7) Prepare data for fine-tuning
def prepare(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16000).input_features[0]
    batch["labels"] = tokenizer(batch[TEXT_COL]).input_ids
    return batch

train_ds = ds["train"].map(prepare, remove_columns=ds["train"].column_names)
dev_ds   = concatenate_datasets([ds["dev_cv"], ds["dev_parl"]]).map(
    prepare, remove_columns=ds["dev_cv"].column_names)

# shrink dev for fast in-training evaluation (the real test sets are unaffected)
dev_ds = dev_ds.shuffle(seed=42).select(range(300))

print("train:", len(train_ds), "| dev (subset):", len(dev_ds))

In [ ]:
# 8) Data collator + metrik
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class Collator:
    processor: Any
    def __call__(self, features):
        inp = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(inp, return_tensors="pt")
        lab = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(lab, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = Collator(processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_str = [normalize(x) for x in pred_str]
    label_str = [normalize(x) for x in label_str]
    return {"wer": 100*jiwer.wer(label_str, pred_str),
            "cer": 100*jiwer.cer(label_str, pred_str)}

In [ ]:
# 9) Training (free-T4 settings)
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

model = load_model()
model.config.use_cache = False

args = Seq2SeqTrainingArguments(
    output_dir=f"{SAVE}/whisper-small-fi-tokens",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch = 16
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=800,                      # increase/decrease for T4 time budget
    gradient_checkpointing=True,
    fp16=True,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    eval_strategy="steps",
    eval_steps=400,
    save_steps=400,                     # same as eval_steps
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
trainer.train()
trainer.save_model(f"{SAVE}/whisper-final")
print("Model saved to Drive ✓")

In [ ]:
# 10) FINE-TUNED model → each of the 3 test sets separately
model.config.use_cache = True
finetuned_rows = report(model, "fine-tuned")

In [ ]:
# 11) Comparison table (baseline vs fine-tuned)
print(f"{'Test set':12s} | {'Baseline WER':>12s} {'CER':>6s} | {'FT WER':>7s} {'CER':>6s}")
print("-"*60)
for (s, bw, bc), (_, fw, fc) in zip(baseline_rows, finetuned_rows):
    print(f"{s:12s} | {bw:12.2f} {bc:6.2f} | {fw:7.2f} {fc:6.2f}")

In [ ]:
# When finished: write results to CSV + zip the model + download
import csv
from google.colab import files

# 1) Results table -> results.csv
with open("results.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["test_set","baseline_WER","baseline_CER","finetuned_WER","finetuned_CER"])
    for (s, bw, bc), (_, fw, fc) in zip(baseline_rows, finetuned_rows):
        w.writerow([s, round(bw,2), round(bc,2), round(fw,2), round(fc,2)])

# 2) Save model + zip
trainer.save_model("whisper-final")
!zip -r -q whisper-final.zip whisper-final

# 3) Download to your computer
files.download("results.csv")
files.download("whisper-final.zip")

In [ ]:
# 12) (optional) Push the model to the Hugging Face Hub
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub("tia88na/whisper-small-fi-tokens")

In [ ]:
import csv, os
os.makedirs("preds", exist_ok=True)

@torch.no_grad()
def save_predictions(model, split, tag, batch_size=16):
    model.eval()
    if DEVICE=="cuda": model.half()
    dset = ds[split]; rows=[]
    for i in tqdm(range(0, len(dset), batch_size), desc=split):
        batch = dset[i:i+batch_size]
        arrays = [a["array"] for a in batch["audio"]]
        feats = processor.feature_extractor(arrays, sampling_rate=16000,
                 return_tensors="pt").input_features.to(DEVICE)
        if DEVICE=="cuda": feats = feats.half()
        gen = model.generate(feats, max_new_tokens=225)
        pred = processor.batch_decode(gen, skip_special_tokens=True)
        for j in range(len(pred)):
            rows.append({"id": batch["id"][j],
                         "reference": normalize(batch[TEXT_COL][j]),
                         "hypothesis": normalize(pred[j])})
    path=f"preds/{tag}_{split}.csv"
    with open(path,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=["id","reference","hypothesis"])
        w.writeheader(); w.writerows(rows)
    print("wrote:", path, len(rows), "rows")

for s in ["test_cv","test_parl","test_fleurs"]:
    save_predictions(model, s, "finetuned")

!zip -r -q preds.zip preds
from google.colab import files
files.download("preds.zip")


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
model.push_to_hub("tia88na/whisper-small-fi-tokens")
processor.push_to_hub("tia88na/whisper-small-fi-tokens")